
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F5839C6660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F5839C7380>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
from pydantic import BaseModel,Field

class Recipe(BaseModel):
    name:str=Field(description="The name of the dish")
    cuisine:str=Field(description="The cuisine style of the dish")
    prep_time:int=Field(description="Preparation time in minutes")
    ingredients:list[str]=Field(description="List of ingredients needed for the dish")

In [ ]:
model_with_structure=model.with_structured_output(Recipe)
model_with_structure

In [ ]:
model.invoke("Tell me about khachapuri, the Georgian cheese-filled bread")

In [ ]:
response=model_with_structure.invoke("Provide the recipe details for khachapuri, the Georgian cheese-filled bread")
response

### MEssage output alongside parsed structure

In [ ]:
from pydantic import BaseModel, Field

class Recipe(BaseModel):
    """A recipe with details."""
    name: str = Field(..., description="The name of the dish")
    cuisine: str = Field(..., description="The cuisine style of the dish")
    prep_time: int = Field(..., description="Preparation time in minutes")
    ingredients: list[str] = Field(..., description="List of ingredients needed for the dish")

model_with_structure = model.with_structured_output(Recipe, include_raw=True)  

response = model_with_structure.invoke("Provide the recipe details for khachapuri, the Georgian cheese-filled bread")
response

### Nested Structure

In [ ]:
from pydantic import BaseModel, Field

class Ingredient(BaseModel):
    item: str
    quantity: str

class RecipeDetails(BaseModel):
    name: str
    cuisine: str
    ingredients: list[Ingredient]
    steps: list[str]
    prep_time: float | None = Field(None, description="Prep time in minutes")

model_with_structure = model.with_structured_output(RecipeDetails)

response = model_with_structure.invoke("Provide the full recipe for khachapuri, the Georgian cheese-filled bread")
response

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [ ]:
from typing_extensions import TypedDict,Annotated

class RecipeDict(TypedDict):
    """A recipe with details."""
    name: Annotated[str, ..., "The name of the dish"]
    cuisine: Annotated[str, ..., "The cuisine style of the dish"]
    prep_time: Annotated[int, ..., "Preparation time in minutes"]
    ingredients: Annotated[list[str], ..., "List of ingredients needed for the dish"]


model_withtypedict=model.with_structured_output(RecipeDict)
response=model_withtypedict.invoke("Please provide the recipe details for sinigang")
response

In [ ]:
class Ingredient(TypedDict):
    item: str
    quantity: str

class RecipeDetails(TypedDict):
    name: str
    cuisine: str
    ingredients: list[Ingredient]
    steps: list[str]
    prep_time: float | None = Field(None, description="Prep time in minutes")

model_with_structure = model.with_structured_output(RecipeDetails)

response = model_with_structure.invoke("Provide the full recipe for sinigang")
response

In [21]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [ ]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029"}]
})

result

In [ ]:
result["structured_response"]

In [ ]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029"}]
})

result["structured_response"]
# {'name': 'Elena Marquez', 'email': 'elena.marquez@brightleaf-consulting.com', 'phone': '(312) 774-5029'}

In [ ]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029"}]
})

result["structured_response"]

### 🤔 Before you move on
You now have an agent that calls tools AND returns clean, guaranteed-shape output. But what happens when a conversation runs long enough to blow past the model's context window, or when a tool call is risky enough that it should pause and wait for a human to approve it first? Neither is solved yet — that's exactly what `6-middleware.ipynb` adds.